# In Class Activity April 21st, 2026

In [ ]:
# importing libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier


In [2]:
# importing the data
adult = pd.read_csv('/Users/christinewu/Downloads/adult.csv')
adult.head()


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


### Model-agnostic preprocessing & fixed train/test split

In [3]:
# replace ? with np.nan and standardize string columns
adult = adult.replace('?', np.nan)

for col in adult.select_dtypes(include='object').columns:
    adult[col] = adult[col].str.strip()

# convert target to binary
adult['income'] = adult['income'].map({'<=50K': 0, '>50K': 1})

# convert gender to 0/1 to match the style used in the demo notebook
adult['gender'] = adult['gender'].map({'Female': 0, 'Male': 1})

adult.head()


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0


In [4]:
# fixed train/test split for the entire notebook
train_idx, test_idx = train_test_split(
    adult.index,
    test_size=0.2,
    stratify=adult['income'],
    random_state=42
)

print('Train rows:', len(train_idx))
print('Test rows:', len(test_idx))


Train rows: 39073
Test rows: 9769


### Feature engineering library

In [5]:
def add_features(df):
    df = df.copy()

    # numeric transformations
    df['capital_total'] = df['capital-gain'] + df['capital-loss']
    df['net_capital'] = df['capital-gain'] - df['capital-loss']
    df['has_capital_gain'] = (df['capital-gain'] > 0).astype(int)
    df['has_capital_loss'] = (df['capital-loss'] > 0).astype(int)
    df['log_fnlwgt'] = np.log1p(df['fnlwgt'])
    df['log_capital_total'] = np.log1p(df['capital_total'])
    df['age_sq'] = df['age'] ** 2
    df['age_x_hours'] = df['age'] * df['hours-per-week']
    df['education_x_age'] = df['educational-num'] * df['age']
    df['capital_per_hour'] = df['capital_total'] / (df['hours-per-week'] + 1)

    # grouped categories
    df['marital_group'] = np.where(
        df['marital-status'].isin(['Married-civ-spouse', 'Married-AF-spouse']),
        'Married',
        np.where(
            df['marital-status'].isin(['Divorced', 'Separated', 'Widowed']),
            'Previously-married',
            'Not-married'
        )
    )

    white_collar = ['Exec-managerial', 'Prof-specialty', 'Tech-support', 'Sales', 'Adm-clerical']
    blue_collar = ['Craft-repair', 'Machine-op-inspct', 'Transport-moving', 'Handlers-cleaners', 'Farming-fishing']
    service = ['Other-service', 'Priv-house-serv', 'Protective-serv']

    df['occupation_group'] = np.where(
        df['occupation'].isin(white_collar),
        'White-collar',
        np.where(
            df['occupation'].isin(blue_collar),
            'Blue-collar',
            np.where(df['occupation'].isin(service), 'Service', 'Other')
        )
    )

    df['country_group'] = np.where(df['native-country'] == 'United-States', 'US', 'Non-US')

    # bucketed features for interpretability
    df['age_band'] = pd.cut(
        df['age'],
        bins=[0, 25, 35, 45, 55, 65, 100],
        labels=['<=25', '26-35', '36-45', '46-55', '56-65', '65+']
    )

    df['hours_band'] = pd.cut(
        df['hours-per-week'],
        bins=[0, 25, 40, 50, 60, 100],
        labels=['<=25', '26-40', '41-50', '51-60', '60+']
    )

    return df

adult_fe = add_features(adult)
adult_fe.head()


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,capital_total,net_capital,has_capital_gain,has_capital_loss,log_fnlwgt,log_capital_total,age_sq,age_x_hours,education_x_age,capital_per_hour,marital_group,occupation_group,country_group,age_band,hours_band
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0,0,0,0,0,12.331837,0.000000,625,1000,175,0.000000,Not-married,Blue-collar,US,<=25,26-40
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0,0,0,0,0,11.405507,0.000000,1444,1900,342,0.000000,Married,Blue-collar,US,36-45,41-50
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1,0,0,0,0,12.727696,0.000000,784,1120,336,0.000000,Married,Service,US,26-35,26-40
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1,7688,7688,1,0,11.984952,8.947546,1936,1760,440,187.512195,Married,Blue-collar,US,36-45,26-40
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0,0,0,0,0,11.547308,0.000000,324,540,180,0.000000,Not-married,Other,US,<=25,26-40


In [7]:
# engineered features
engineered_cols = [
    'capital_total', 'net_capital', 'has_capital_gain', 'has_capital_loss',
    'log_fnlwgt', 'log_capital_total', 'age_sq', 'age_x_hours',
    'education_x_age', 'capital_per_hour', 'marital_group',
    'occupation_group', 'country_group', 'age_band', 'hours_band'
]

adult_fe[engineered_cols].head()


,capital_total,net_capital,has_capital_gain,has_capital_loss,log_fnlwgt,log_capital_total,age_sq,age_x_hours,education_x_age,capital_per_hour,marital_group,occupation_group,country_group,age_band,hours_band
0,0,0,0,0,12.331837,0.000000,625,1000,175,0.000000,Not-married,Blue-collar,US,<=25,26-40
1,0,0,0,0,11.405507,0.000000,1444,1900,342,0.000000,Married,Blue-collar,US,36-45,41-50
2,0,0,0,0,12.727696,0.000000,784,1120,336,0.000000,Married,Service,US,26-35,26-40
3,7688,7688,1,0,11.984952,8.947546,1936,1760,440,187.512195,Married,Blue-collar,US,36-45,26-40
4,0,0,0,0,11.547308,0.000000,324,540,180,0.000000,Not-married,Other,US,<=25,26-40


### Model 1: Logistic regression with one-hot encoding and scaling

In [8]:
# train/test data for logistic regression path
log_df = adult_fe.drop(columns=['income']).copy()
y = adult_fe['income'].copy()

X_train_log = log_df.loc[train_idx].copy()
X_test_log = log_df.loc[test_idx].copy()
y_train = y.loc[train_idx].copy()
y_test = y.loc[test_idx].copy()

categorical_cols_log = X_train_log.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols_log = [col for col in X_train_log.columns if col not in categorical_cols_log]

print('Categorical columns for logistic model:', len(categorical_cols_log))
print('Numeric columns for logistic model:', len(numeric_cols_log))


Categorical columns for logistic model: 12
Numeric columns for logistic model: 17


In [10]:
# preprocessing pipeline and baseline logistic regression
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

log_preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols_log),
    ('cat', categorical_pipe, categorical_cols_log)
])

log_baseline = Pipeline([
    ('prep', log_preprocessor),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_cv_scores = cross_val_score(log_baseline, X_train_log, y_train, cv=kf, scoring='roc_auc')
print(f'Baseline Logistic CV ROC AUC: {log_cv_scores.mean():.4f} +/- {log_cv_scores.std():.4f}')

log_baseline.fit(X_train_log, y_train)
log_base_probs = log_baseline.predict_proba(X_test_log)[:, 1]
log_base_pred = (log_base_probs >= 0.5).astype(int)

print('Baseline Logistic Test Accuracy:', round(accuracy_score(y_test, log_base_pred), 4))
print('Baseline Logistic Test ROC AUC:', round(roc_auc_score(y_test, log_base_probs), 4))
print('Baseline Logistic Classification Report:', classification_report(y_test, log_base_pred))


Baseline Logistic CV ROC AUC: 0.9140 +/- 0.0034
Baseline Logistic Test Accuracy: 0.8596
Baseline Logistic Test ROC AUC: 0.9141
Baseline Logistic Classification Report:               precision    recall  f1-score   support

           0       0.89      0.94      0.91      7431
           1       0.75      0.61      0.68      2338

    accuracy                           0.86      9769
   macro avg       0.82      0.78      0.79      9769
weighted avg       0.85      0.86      0.85      9769



In [12]:
# tuning logistic regression beyond defaults
log_tuned = Pipeline([
    ('prep', log_preprocessor),
    ('model', LogisticRegression(max_iter=3000, solver='saga', random_state=42))
])

log_param_grid = {
    'model__C': np.logspace(-2, 1.5, 8),
    'model__penalty': ['l1', 'l2'],
    'model__class_weight': [None, 'balanced']
}

log_search = RandomizedSearchCV(
    estimator=log_tuned,
    param_distributions=log_param_grid,
    n_iter=10,
    scoring='roc_auc',
    cv=kf,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

log_search.fit(X_train_log, y_train)
print('Best Logistic Params:', log_search.best_params_)
print('Best Logistic CV ROC AUC:', round(log_search.best_score_, 4))

best_log = log_search.best_estimator_
log_probs = best_log.predict_proba(X_test_log)[:, 1]
log_pred = (log_probs >= 0.5).astype(int)

print('Tuned Logistic Test Accuracy:', round(accuracy_score(y_test, log_pred), 4))
print('Tuned Logistic Test ROC AUC:', round(roc_auc_score(y_test, log_probs), 4))
print('Tuned Logistic Classification Report:', classification_report(y_test, log_pred))


Fitting 5 folds for each of 10 candidates, totalling 50 fits


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max

Best Logistic Params: {'model__penalty': 'l2', 'model__class_weight': 'balanced', 'model__C': np.float64(0.31622776601683794)}
Best Logistic CV ROC AUC: 0.9141
Tuned Logistic Test Accuracy: 0.8163
Tuned Logistic Test ROC AUC: 0.9142
Tuned Logistic Classification Report:               precision    recall  f1-score   support

           0       0.94      0.81      0.87      7431
           1       0.58      0.85      0.69      2338

    accuracy                           0.82      9769
   macro avg       0.76      0.83      0.78      9769
weighted avg       0.86      0.82      0.83      9769



In [13]:
# logistic feature effects (largest coefficients by magnitude)
feature_names_log = best_log.named_steps['prep'].get_feature_names_out()
coef_values = best_log.named_steps['model'].coef_[0]

log_coef_df = pd.DataFrame({
    'feature': feature_names_log,
    'coefficient': coef_values,
    'abs_coefficient': np.abs(coef_values)
}).sort_values('abs_coefficient', ascending=False)

print('Top positive logistic coefficients:')
print(log_coef_df.sort_values('coefficient', ascending=False).head(10))

print('Top negative logistic coefficients:')
print(log_coef_df.sort_values('coefficient', ascending=True).head(10))


Top positive logistic coefficients:
                             feature  coefficient  abs_coefficient
0                           num__age     2.194331         2.194331
12            num__log_capital_total     1.492419         1.492419
7                 num__capital_total     1.486261         1.486261
4                  num__capital-gain     1.447281         1.447281
8                   num__net_capital     1.403967         1.403967
58   cat__occupation_Protective-serv     0.980592         0.980592
114       cat__marital_group_Married     0.912630         0.912630
67            cat__relationship_Wife     0.840760         0.840760
81       cat__native-country_England     0.779913         0.779913
93       cat__native-country_Ireland     0.726682         0.726682
Top negative logistic coefficients:
                                    feature  coefficient  abs_coefficient
9                     num__has_capital_gain    -1.698325         1.698325
13                              num__age_sq

### Model 2: XGBoost with native categorical support

In [14]:
# train/test data for XGBoost native categorical path
xgb_df = adult_fe.copy()

for col in xgb_df.select_dtypes(include='object').columns:
    xgb_df[col] = xgb_df[col].fillna('Unknown').astype('category')

X_xgb = xgb_df.drop(columns=['income']).copy()
y_xgb = xgb_df['income'].copy()

X_train_xgb = X_xgb.loc[train_idx].copy()
X_test_xgb = X_xgb.loc[test_idx].copy()
y_train_xgb = y_xgb.loc[train_idx].copy()
y_test_xgb = y_xgb.loc[test_idx].copy()

X_train_xgb.dtypes.head()


age                   int64
workclass          category
fnlwgt                int64
education          category
educational-num       int64
dtype: object

In [16]:
# baseline XGBoost model with native categorical handling
xgb_baseline = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    enable_categorical=True,
    tree_method='hist',
    n_estimators=250,
    learning_rate=0.08,
    max_depth=6
)

xgb_cv_scores = cross_val_score(xgb_baseline, X_train_xgb, y_train_xgb, cv=kf, scoring='roc_auc')
print(f'Baseline XGB CV ROC AUC: {xgb_cv_scores.mean():.4f} +/- {xgb_cv_scores.std():.4f}')

xgb_baseline.fit(X_train_xgb, y_train_xgb)
xgb_base_probs = xgb_baseline.predict_proba(X_test_xgb)[:, 1]
xgb_base_pred = (xgb_base_probs >= 0.5).astype(int)

print('Baseline XGB Test Accuracy:', round(accuracy_score(y_test_xgb, xgb_base_pred), 4))
print('Baseline XGB Test ROC AUC:', round(roc_auc_score(y_test_xgb, xgb_base_probs), 4))
print('Baseline XGB Classification Report:', classification_report(y_test_xgb, xgb_base_pred))


Baseline XGB CV ROC AUC: 0.9283 +/- 0.0025
Baseline XGB Test Accuracy: 0.8747
Baseline XGB Test ROC AUC: 0.9302
Baseline XGB Classification Report:               precision    recall  f1-score   support

           0       0.90      0.94      0.92      7431
           1       0.78      0.66      0.72      2338

    accuracy                           0.87      9769
   macro avg       0.84      0.80      0.82      9769
weighted avg       0.87      0.87      0.87      9769



In [18]:
# tuning XGBoost beyond defaults
xgb_tuned = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    enable_categorical=True,
    tree_method='hist'
)

xgb_param_grid = {
    'n_estimators': [150, 250, 350],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.03, 0.05, 0.08, 0.12],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'min_child_weight': [1, 3, 5],
    'reg_lambda': [1, 2, 5]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_tuned,
    param_distributions=xgb_param_grid,
    n_iter=12,
    scoring='roc_auc',
    cv=kf,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train_xgb, y_train_xgb)
print('Best XGB Params:', xgb_search.best_params_)
print('Best XGB CV ROC AUC:', round(xgb_search.best_score_, 4))

best_xgb = xgb_search.best_estimator_
xgb_probs = best_xgb.predict_proba(X_test_xgb)[:, 1]
xgb_pred = (xgb_probs >= 0.5).astype(int)

print('Tuned XGB Test Accuracy:', round(accuracy_score(y_test_xgb, xgb_pred), 4))
print('Tuned XGB Test ROC AUC:', round(roc_auc_score(y_test_xgb, xgb_probs), 4))
print('Tuned XGB Classification Report:', classification_report(y_test_xgb, xgb_pred))


Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best XGB Params: {'subsample': 1.0, 'reg_lambda': 2, 'n_estimators': 350, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.08, 'colsample_bytree': 0.7}
Best XGB CV ROC AUC: 0.9285
Tuned XGB Test Accuracy: 0.8761
Tuned XGB Test ROC AUC: 0.93
Tuned XGB Classification Report:               precision    recall  f1-score   support

           0       0.90      0.94      0.92      7431
           1       0.79      0.66      0.72      2338

    accuracy                           0.88      9769
   macro avg       0.84      0.80      0.82      9769
weighted avg       0.87      0.88      0.87      9769



In [19]:
# XGBoost feature importance
xgb_importance_df = pd.DataFrame({
    'feature': best_xgb.feature_names_in_,
    'importance': best_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 15 XGB features:')
print(xgb_importance_df.head(15))


Top 15 XGB features:
              feature  importance
7        relationship    0.196122
5      marital-status    0.140137
24      marital_group    0.119807
19  log_capital_total    0.082721
4     educational-num    0.074132
14      capital_total    0.067128
10       capital-gain    0.041939
6          occupation    0.035104
16   has_capital_gain    0.031600
22    education_x_age    0.024878
3           education    0.023590
11       capital-loss    0.021051
15        net_capital    0.016681
9              gender    0.015416
27           age_band    0.015128


### Probability averaging ensemble

In [21]:
# equal-weight probability average ensemble
avg_probs_equal = (log_probs + xgb_probs) / 2
avg_pred_equal = (avg_probs_equal >= 0.5).astype(int)

print('Equal-weight Ensemble Accuracy:', round(accuracy_score(y_test, avg_pred_equal), 4))
print('Equal-weight Ensemble ROC AUC:', round(roc_auc_score(y_test, avg_probs_equal), 4))
print('Equal-weight Ensemble Classification Report:', classification_report(y_test, avg_pred_equal))


Equal-weight Ensemble Accuracy: 0.863
Equal-weight Ensemble ROC AUC: 0.9259
Equal-weight Ensemble Classification Report:               precision    recall  f1-score   support

           0       0.92      0.90      0.91      7431
           1       0.70      0.76      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.81      0.83      0.82      9769
weighted avg       0.87      0.86      0.87      9769



In [22]:
# optional weighted probability average
avg_probs_weighted = 0.4 * log_probs + 0.6 * xgb_probs
avg_pred_weighted = (avg_probs_weighted >= 0.5).astype(int)

print('Weighted Ensemble Accuracy:', round(accuracy_score(y_test, avg_pred_weighted), 4))
print('Weighted Ensemble ROC AUC:', round(roc_auc_score(y_test, avg_probs_weighted), 4))


Weighted Ensemble Accuracy: 0.8685
Weighted Ensemble ROC AUC: 0.9272


### Side-by-side comparison

In [23]:
# compare all main results in one table
results_df = pd.DataFrame({
    'model': [
        'Baseline Logistic',
        'Tuned Logistic',
        'Baseline XGBoost',
        'Tuned XGBoost',
        'Equal-weight Ensemble',
        'Weighted Ensemble'
    ],
    'accuracy': [
        accuracy_score(y_test, log_base_pred),
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test_xgb, xgb_base_pred),
        accuracy_score(y_test_xgb, xgb_pred),
        accuracy_score(y_test, avg_pred_equal),
        accuracy_score(y_test, avg_pred_weighted)
    ],
    'roc_auc': [
        roc_auc_score(y_test, log_base_probs),
        roc_auc_score(y_test, log_probs),
        roc_auc_score(y_test_xgb, xgb_base_probs),
        roc_auc_score(y_test_xgb, xgb_probs),
        roc_auc_score(y_test, avg_probs_equal),
        roc_auc_score(y_test, avg_probs_weighted)
    ]
}).sort_values('roc_auc', ascending=False)

results_df


,model,accuracy,roc_auc
2,Baseline XGBoost,0.874706,0.930182
3,Tuned XGBoost,0.876139,0.929950
5,Weighted Ensemble,0.868461,0.927166
4,Equal-weight Ensemble,0.863036,0.925916
1,Tuned Logistic,0.816256,0.914237
0,Baseline Logistic,0.859556,0.914138


### Evaluation of Results

The baseline XGBoost model achieved the best overall performance, with an accuracy of 0.875 and a ROC AUC of 0.930. Feature engineering played an important role by combining transformations, interactions, and grouped categories, which provided a more informative representation of the data. Logistic regression highlighted strong linear relationships, with age and capital-related features being the most influential. In contrast, XGBoost emphasized categorical variables such as relationship and marital status, showing its ability to capture nonlinear patterns. Model tuning improved performance compared to baseline settings, and the comparison between models suggests they respond differently to the same engineered features. Probability averaging offers a way to test whether combining these strengths leads to further gains, and future iterations should retain the most impactful engineered features while removing those with limited contribution.